## Explanation
This code is used to create local convexity maps of fans

In [ ]:
!pwd

import os
os.chdir("../../")

In [ ]:
import numpy as np
from sympy import symbols, Eq, diff, solve, Matrix, lambdify
from ruamel.yaml import YAML
from matplotlib.colors import ListedColormap, BoundaryNorm

from underestimating_hyperplanes import get_pel_n_func, calculate_max_values, sample_over_grid_nd
from src.preprocessing.domain_utils import dimensionalise_coeffs, get_max_values

to ensure that the figures look like in the publication, a certain style-file is loaded

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("FST.mplstyle")

pt = 1./72.27
diss_tex_width = 390.0*pt

from matplotlib.font_manager import FontProperties
arial_font = FontProperties(family="Arial", style="italic")

## Functions to compute Hessian and thus convexity (by Sylvester's criterion)

In [ ]:
inverse_dp_n_lam_a = lambda q,dp: inverse_dp_n(a,q,dp)

Hess_pel_ = lambda dp, q:   Hess_pel(   q,dp,a[1],a[2],a[3],b[1],b[2],b[3],b[4])
Hess_ploss_ = lambda dp, q: Hess_ploss( q,dp,a[1],a[2],a[3],b[1],b[2],b[3],b[4])


def calculate_Hess():

    pel, q, dp, n, g = symbols('pel q dp n g')
    a = symbols('a1:4')
    b = symbols('b1:5')

    H = Eq(dp, a[0]*q**2 + a[1]*q*n + a[2]*n**2)
    pel = b[0]*q**3 + b[1]*q**2*n + b[2]*q*n**2 + b[3]*n**3

    n_eq = solve(H, n)[1]
    pel_eq = pel.subs(n, n_eq)

    ploss = b[0]*q**3 + b[1]*q**2*n + b[2]*q*n**2 + b[3]*n**3 - q*dp
    ploss_eq = ploss.subs(n, n_eq)

    # Compute second order partial derivatives
    f_complex_xx = diff(diff(pel_eq, q), q)
    f_complex_xy = diff(diff(pel_eq, q), dp)
    f_complex_yy = diff(diff(pel_eq, dp), dp)

    Hess_pel = Matrix([[f_complex_xx, f_complex_xy],
                [f_complex_xy, f_complex_yy]])

    Hess_pel = lambdify([q, dp, *a, *b], Hess_pel, modules="numpy")

    # Compute second order partial derivatives
    f_complex_xx = diff(diff(ploss_eq, q), q)
    f_complex_xy = diff(diff(ploss_eq, q), dp)
    f_complex_yy = diff(diff(ploss_eq, dp), dp)

    Hess_ploss = Matrix([[f_complex_xx, f_complex_xy],
                [f_complex_xy, f_complex_yy]])

    Hess_ploss = lambdify([q, dp, *a, *b], Hess_ploss, modules="numpy")

    return Hess_pel, Hess_ploss

def calculate_principal_minors(Hess):
    return int(Hess[0,0]>=0) + 2*int(Hess[1,1]>= 0) + 4*int(np.linalg.det(Hess)>=0)

def inverse_dp_n(a,q,dp):
    return (-a[2]*q + np.sqrt(-4*a[1]*a[3]*q**2 + a[2]**2*q**2 + 4*a[3]*dp))/(2*a[3])


def compute_local_convexity(dps, qs, Hess_pel_):
    convexity_matrix = np.nan*np.ones([len(qs),len(dps)])

    for idx,dpi in enumerate(dps):
        for jdx,qi in enumerate(qs):

            ni = inverse_dp_n_lam_a(qi, dpi)
            if ni <= 1:
                convexity_matrix[idx,jdx] = calculate_principal_minors(Hess_pel_(dpi,qi)) == 7
    return convexity_matrix

def plot_convexity(ax, convexity_matrix):
    # Define the fixed colors for NaN, 1, 2, 3, 4, 5, 6, 7
    colors = [[220/255]*3, [137/255]*3]
    cmap = ListedColormap(colors)

    # Define boundaries so each value gets its exact color
    bounds = [-0.5, 0.5, 1.5]
    norm = BoundaryNorm(bounds, cmap.N)


    ax.imshow(convexity_matrix, extent=[qs.min(), qs.max(), dps.min(), dps.max()], origin='lower', aspect='auto', cmap=cmap, norm=norm)#, cmap=binary_cmap)
    ax.set_xlabel("VOLUMENSTROM")
    ax.set_ylabel("DRUCKERHÖHUNG")

    ax.set_xticks([])
    ax.set_yticks([])

    return ax

def compute_convexity_ratio(convexity_matrix):
    return np.sum(convexity_matrix==1) / ( np.sum(convexity_matrix == 0) + np.sum(convexity_matrix == 1) )


compute Hessian as a function of $q$, $\Delta p$

In [ ]:
Hess_pel, Hess_ploss = calculate_Hess()


In [ ]:
def load_yaml_config(path):
    yaml = YAML(typ="safe")  # or typ="rt" if you want to preserve formatting
    with open(path, 'r') as f:
        data = yaml.load(f)
    return data

fan = "Fan2"
d = 0.4

fan_data = load_yaml_config("data/fan_data/%s.yml"%fan)

while the input is the electrical power consumption. in get_pel_n_func, the function p_loss is computed

In [ ]:
alpha = fan_data["ansatz"]["pressure"]["o2"]["ansatz_param"]
beta = fan_data["ansatz"]["power"]["o3"]["ansatz_param"]

max_values = get_max_values(fan_data)

a, b = dimensionalise_coeffs(alpha, beta, d, max_values)



pel_expr = lambda q, n: b[1] * q**3 + b[2] * q**2*n + b[3] * q*n**2 + b[4]*n**3 + b[5]
pel_expr_grad = lambda q, n: 3*b[1]*q**2 + 2*b[2]*q*n + b[3]*n**2

ploss_func, ploss_func_grad, n_func = get_pel_n_func(a,b)

max_values = calculate_max_values(a, b)
Qmax, dpmax, Pmax = max_values.values()


dps = np.linspace(0,dpmax,100)
qs = np.linspace(0,Qmax,100)

In [ ]:
print(Qmax, dpmax, Pmax)

In [ ]:
q_vals = np.linspace(Qmax * 1e-9, Qmax, 20)
dp_vals = np.linspace(dpmax * 1e-9, dpmax, 20)

# n_func works as a constraint here, so that n(q,dp)<=1
points = sample_over_grid_nd([q_vals, dp_vals], n_func, )

q_vals = np.linspace(0, Qmax, 20)
dp_vals = np.linspace(0, dpmax, 20)

finer_points = sample_over_grid_nd([q_vals, dp_vals], n_func)

len(points)

In [ ]:
convexity_matrix = compute_local_convexity(dps, qs, Hess_ploss_)
fig,ax = plt.subplots(1,1, figsize=(0.5*diss_tex_width,diss_tex_width*3/9))

ax = plot_convexity(ax, convexity_matrix)

fig.tight_layout()
fig.savefig("plots/diss/convexity_power_loss_%s.svg"%fan)

In [ ]:
convexity_matrix = compute_local_convexity(dps, qs, Hess_pel_)
fig,ax = plt.subplots(1,1, figsize=(0.5*diss_tex_width,diss_tex_width*3/9))

ax = plot_convexity(ax, convexity_matrix)

fig.tight_layout()
fig.savefig("plots/diss/convexity_power_consumption_%s.svg"%fan)